# Book Recommendation Engine using KNN

Completed solution for the freeCodeCamp **Machine Learning with Python** project.

Run the notebook from top to bottom. The final cell is the original freeCodeCamp test.


In [ ]:
# import libraries (you may add additional imports but you may not have to)
import os
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# get data files
# Make the download cell safe to re-run in Colab.
!rm -f book-crossings.zip BX-Books.csv BX-Book-Ratings.csv BX-Users.csv
!wget -q https://cdn.freecodecamp.org/project-data/books/book-crossings.zip -O book-crossings.zip
!unzip -oq book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

In [ ]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding="ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'}
)

df_ratings = pd.read_csv(
    ratings_filename,
    encoding="ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'}
)

print("Books:", len(df_books))
print("Ratings:", len(df_ratings))

In [ ]:
# Keep statistically meaningful users and books.
# The challenge says to remove users with < 200 ratings and books with < 100 ratings.
user_counts = df_ratings['user'].value_counts()
active_users = user_counts[user_counts >= 200].index

book_counts = df_ratings['isbn'].value_counts()
popular_isbns = book_counts[book_counts >= 100].index

filtered_ratings = df_ratings[
    df_ratings['user'].isin(active_users)
    & df_ratings['isbn'].isin(popular_isbns)
].copy()

# Build the book x user ratings matrix. ISBN is used as the row key so different
# editions with the same title do not get accidentally merged.
ratings_matrix = filtered_ratings.pivot_table(
    index='isbn',
    columns='user',
    values='rating',
    aggfunc='mean',
    fill_value=0
)

ratings_sparse = csr_matrix(ratings_matrix.values)

# Metadata restricted to ISBNs that actually exist in the model matrix.
model_books = (
    df_books[df_books['isbn'].isin(ratings_matrix.index)]
    .drop_duplicates(subset='isbn')
    .copy()
)

isbn_to_title = model_books.set_index('isbn')['title'].to_dict()

# A title can have multiple editions. For a title query, select the eligible
# ISBN with the largest number of non-zero ratings in the filtered matrix.
title_to_isbns = (
    model_books.groupby('title')['isbn']
    .apply(list)
    .to_dict()
)

def _isbn_for_title(title):
    candidates = [
        isbn for isbn in title_to_isbns.get(title, [])
        if isbn in ratings_matrix.index
    ]
    if not candidates:
        return None

    return max(
        candidates,
        key=lambda isbn: int(np.count_nonzero(ratings_matrix.loc[isbn].to_numpy()))
    )

print("Active users:", len(active_users))
print("Popular ISBNs:", len(popular_isbns))
print("Model matrix:", ratings_matrix.shape)

In [ ]:
# Train KNN using cosine distance.
model_knn = NearestNeighbors(
    metric='cosine',
    algorithm='brute'
)
model_knn.fit(ratings_sparse)

In [ ]:
# function to return recommended books - this will be tested
def get_recommends(book=""):
    query_isbn = _isbn_for_title(book)

    if query_isbn is None:
        return [book, []]

    query_position = ratings_matrix.index.get_loc(query_isbn)
    query_vector = ratings_sparse[query_position]

    # Query a few extra rows in case another edition has the same title.
    neighbor_count = min(12, ratings_matrix.shape[0])
    distances, indices = model_knn.kneighbors(
        query_vector,
        n_neighbors=neighbor_count
    )

    recommendations = []

    for distance, index in zip(distances.flatten(), indices.flatten()):
        candidate_isbn = ratings_matrix.index[index]

        # Exclude the exact queried edition.
        if candidate_isbn == query_isbn:
            continue

        candidate_title = isbn_to_title.get(candidate_isbn)
        if candidate_title is None:
            continue

        # Avoid returning the queried title under another ISBN and avoid
        # duplicate titles in the five recommendations.
        if candidate_title == book:
            continue
        if any(existing[0] == candidate_title for existing in recommendations):
            continue

        recommendations.append([candidate_title, float(distance)])

        if len(recommendations) == 5:
            break

    # freeCodeCamp's expected examples list these five neighbors in reverse
    # KNN order: farther among the five first, closest last.
    recommendations.reverse()

    return [book, recommendations]

In [ ]:
# Optional example from the project statement.
example = get_recommends(
    "The Queen of the Damned (Vampire Chronicles (Paperback))"
)
print(example)

In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()